In [2]:
!pip install -r requirements.txt

In [34]:
import mlflow
import pickle
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error

Parquet files downloaded from https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [5]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet
!mv green_tripdata_2021-01.parquet data/

--2026-03-29 15:31:19--  https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.163.140.18, 3.163.140.37, 3.163.140.127, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.163.140.18|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1333519 (1.3M) [binary/octet-stream]
Saving to: ‘green_tripdata_2021-01.parquet.1’

green_tripdata_2021 100%[===================>]   1.27M   404KB/s    in 3.2s    

2026-03-29 15:31:23 (404 KB/s) - ‘green_tripdata_2021-01.parquet.1’ saved [1333519/1333519]



In [6]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet
!mv green_tripdata_2021-02.parquet data/

--2026-03-29 15:31:45--  https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.163.140.127, 3.163.140.37, 3.163.140.145, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.163.140.127|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1145679 (1.1M) [binary/octet-stream]
Saving to: ‘green_tripdata_2021-02.parquet’

green_tripdata_2021 100%[===================>]   1.09M  1.01MB/s    in 1.1s    

2026-03-29 15:31:48 (1.01 MB/s) - ‘green_tripdata_2021-02.parquet’ saved [1145679/1145679]



In [7]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("nyc-taxi-experiment")

2026/03/29 15:34:53 INFO mlflow.tracking.fluent: Experiment with name 'nyc-taxi-experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1774809293004, experiment_id='3', last_update_time=1774809293004, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, workspace='default'>

In [32]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)
    
    df = df[df.trip_type == 2]
    
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

In [35]:
df_train = read_dataframe('data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('data/green_tripdata_2021-02.parquet')

In [36]:
categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

target = 'duration'

y_train = df_train[target].values
y_val = df_val[target].values


In [38]:
with mlflow.start_run():

    mlflow.set_tag('developer', 'daniel')
    mlflow.log_param('train-data-path', 'data/green_tripdata_2021-01.parquet')
    mlflow.log_param('val-data-path', 'data/green_tripdata_2021-02.parquet')

    alpha = 0.01
    mlflow.log_param('alpha', alpha)

    lr = Lasso(alpha)
    lr.fit(X_train, y_train)
    
    y_pred = lr.predict(X_val)
    
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric('rmse', rmse)

🏃 View run dapper-ray-769 at: http://localhost:5000/#/experiments/3/runs/dbe22de032b646c9bcd858241aa80a42
🧪 View experiment at: http://localhost:5000/#/experiments/3
